In [11]:
from data_frame.joins.joiner import Joiner
from data_frame.spark_utils import get_spark


In [12]:
spark = get_spark(app_name="Join types")

In [13]:

# Create DataFrames
employees = [
    (1, "Alice", 101),
    (2, "Bob", 102),
    (3, "Charlie", 101),
    (4, "David", 103)
]
df_emp = spark.createDataFrame(employees, ["emp_id", "name", "dept_id"])

departments = [
    (101, "Engineering", "NYC"),
    (102, "Marketing", "LA"),
    (104, "Sales", "CHI")
]
df_dept = spark.createDataFrame(departments, ["dept_id", "dept_name", "location"])

In [14]:
## 1. Different Join Types

In [15]:
# INNER JOIN
inner_result = Joiner.inner_join(df_emp, df_dept, "dept_id")
print("INNER JOIN:")
inner_result.show()

INNER JOIN:
+-------+------+-------+-----------+--------+
|dept_id|emp_id|   name|  dept_name|location|
+-------+------+-------+-----------+--------+
|    101|     3|Charlie|Engineering|     NYC|
|    101|     1|  Alice|Engineering|     NYC|
|    102|     2|    Bob|  Marketing|      LA|
+-------+------+-------+-----------+--------+



In [16]:
# LEFT JOIN
left_result = Joiner.left_join(df_emp, df_dept, "dept_id")
print("LEFT JOIN:")
left_result.show()

LEFT JOIN:
+-------+------+-------+-----------+--------+
|dept_id|emp_id|   name|  dept_name|location|
+-------+------+-------+-----------+--------+
|    101|     1|  Alice|Engineering|     NYC|
|    102|     2|    Bob|  Marketing|      LA|
|    101|     3|Charlie|Engineering|     NYC|
|    103|     4|  David|       NULL|    NULL|
+-------+------+-------+-----------+--------+



In [17]:

# RIGHT JOIN
right_result = Joiner.right_join(df_emp, df_dept, "dept_id")
print("RIGHT JOIN:")
right_result.show()

RIGHT JOIN:
+-------+------+-------+-----------+--------+
|dept_id|emp_id|   name|  dept_name|location|
+-------+------+-------+-----------+--------+
|    101|     3|Charlie|Engineering|     NYC|
|    101|     1|  Alice|Engineering|     NYC|
|    102|     2|    Bob|  Marketing|      LA|
|    104|  NULL|   NULL|      Sales|     CHI|
+-------+------+-------+-----------+--------+



In [18]:

# FULL OUTER JOIN
full_result = Joiner.full_outer_join(df_emp, df_dept, "dept_id")
print("FULL OUTER JOIN:")
full_result.show()

FULL OUTER JOIN:
+-------+------+-------+-----------+--------+
|dept_id|emp_id|   name|  dept_name|location|
+-------+------+-------+-----------+--------+
|    101|     1|  Alice|Engineering|     NYC|
|    101|     3|Charlie|Engineering|     NYC|
|    102|     2|    Bob|  Marketing|      LA|
|    103|     4|  David|       NULL|    NULL|
|    104|  NULL|   NULL|      Sales|     CHI|
+-------+------+-------+-----------+--------+



## 2. Join with Complex Conditions

In [19]:
# Join with condition
condition = (df_emp["dept_id"] == df_dept["dept_id"]) & (df_emp["name"] != "David")
complex_join = Joiner.join_with_condition(df_emp, df_dept, condition, "inner")
print("Join with condition:")
complex_join.show()


Join with condition:
+------+-------+-------+-------+-----------+--------+
|emp_id|   name|dept_id|dept_id|  dept_name|location|
+------+-------+-------+-------+-----------+--------+
|     3|Charlie|    101|    101|Engineering|     NYC|
|     1|  Alice|    101|    101|Engineering|     NYC|
|     2|    Bob|    102|    102|  Marketing|      LA|
+------+-------+-------+-------+-----------+--------+



## 3. Broadcast Join for Optimization

In [20]:
# Create small dimension table
small_dim = spark.createDataFrame([
    (101, "Engineering", "High"),
    (102, "Marketing", "Medium"),
    (103, "Finance", "Low")
], ["dept_id", "dept_name", "priority"])

# Broadcast join (optimized for small tables)
broadcast_result = Joiner.broadcast_join(df_emp, small_dim, "dept_id")
print("Broadcast Join (small table broadcasted):")
broadcast_result.show()

Broadcast Join (small table broadcasted):
+-------+------+-------+-----------+--------+
|dept_id|emp_id|   name|  dept_name|priority|
+-------+------+-------+-----------+--------+
|    101|     1|  Alice|Engineering|    High|
|    102|     2|    Bob|  Marketing|  Medium|
|    101|     3|Charlie|Engineering|    High|
|    103|     4|  David|    Finance|     Low|
+-------+------+-------+-----------+--------+

